# Assignement-3

Q. Fine-tune GPT or GPT-2 for creative story generation

Step 1: Install Dependencies

In [ ]:
!pip install transformers datasets torch accelerate

Step 2: Import Libraries

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments
from datasets import load_dataset
import torch

Step 3: Load Dataset

In [ ]:
dataset = load_dataset("roneneldan/TinyStories")

# Split dataset
train_dataset = dataset['train'].select(range(5000))  # small subset for Colab

Step 4: Load Tokenizer

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

Step 5: Tokenization Function

In [ ]:
def tokenize_function(examples):
    tokenized_inputs = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)
    tokenized_inputs["labels"] = tokenized_inputs["input_ids"].copy() # Add labels for causal language modeling
    return tokenized_inputs

tokenized_dataset = train_dataset.map(tokenize_function, batched=True)

Step 6: Load GPT-2 Model

In [ ]:
model = GPT2LMHeadModel.from_pretrained("gpt2")

Step 7: Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./gpt2-story",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
)

Step 8: Trainer Setup

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

Step 9: Train the Model

In [ ]:
def tokenize_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    tokens["labels"] = tokens["input_ids"].copy()  # 🔥 IMPORTANT LINE
    return tokens

tokenized_dataset = train_dataset.map(tokenize_function, batched=True)

In [24]:
trainer.train()

Step,Training Loss
100,2.246512
200,2.087399
300,2.019876
400,2.004330
500,2.009618
600,1.961510
700,1.969190
800,1.946115
900,1.921062
1000,1.925999


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1250, training_loss=1.9848501953125, metrics={'train_runtime': 306.1734, 'train_samples_per_second': 16.331, 'train_steps_per_second': 4.083, 'total_flos': 326615040000000.0, 'train_loss': 1.9848501953125, 'epoch': 1.0})

Step 10: Save Model

In [25]:
trainer.save_model('./gpt2-story-finetuned')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step 11: Generate Stories

In [26]:
from transformers import pipeline

generator = pipeline("text-generation", model="./gpt2-story-finetuned", tokenizer=tokenizer)

prompt = "Once upon a time in a magical forest"
output = generator(prompt, max_length=100, num_return_sequences=1)

print(output[0]['generated_text'])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Once upon a time in a magical forest, there was a little rabbit named Timmy. Timmy loved to play outside and explore the forest. One day, Timmy's friend, a big brown wolf, came to his house. 

"Hello, Timmy," said the wolf. "Can I come with you?" 

Timmy nodded and said no. But the wolf said, "Will you let me play with you?" 

"Yes, sweetie," said the wolf. But Timmy didn't want to let the wolf play with him. He wanted to play with his friend, his friend, and his friend's friend. 

Timmy felt very sad and scared. But the wolf was not angry. He hugged Timmy and said, "You are my friend, sweetie. I will play with you and hug you together." The wolf smiled and hugged Timmy and said, "Thank you, sweetie. I will always be with you and hug you with my big blue eyes." 

Timmy and the wolf were so happy. They hugged and hugged for a long time. Suddenly, the wolf's friend, a big, friendly frog, came to visit. The wolf said to Timmy, "Hi, Timmy! Can you
